In [ ]:
# =================================================================
# A Novel QRS Detection: Gradient–based Algorithm 
# =================================================================
# This script implements the R-peak detection algorithm described in my FYP report:
# ECG Signal Analysis Algorithm Empowered by AI, and represents its first part.

'''''
Author: Karim Nadjarian Hayek
Email: karimn.hayek@gmail.com
University Email: karimnh@purdue.edu
*****************************************************************************************************
COPYRIGHT: © 2024-2026 ALL RIGHTS RESERVED to the author.   

Neither this notebook nor any part may be reproduced or transmitted in any form or by any means,    
electronic or mechanical, including photocopying, microfilming, and recording, or by any information
storage or retrieval system, without prior permission in writing from the author, the company, or the
university.                   
*****************************************************************************************************
'''''

# =================================================================
# Required packages
# =================================================================
# - scipy.signal
# - numpy
# - math
# - matplotlib.pyplot
# - wfbd python (for reading test data and processing)
# - statsmodels.tsa.api (for exponential smoothing)

# Open source test data is taken from:
# https://physionet.org/content/mitdb/1.0.0/
# https://physionet.org/content/edb/1.0.0/
# https://physionet.org/content/nstdb/1.0.0/

import wfdb
import math
import numpy as np
import scipy.signal as sg
from wfdb import processing
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from statsmodels.tsa.api import SimpleExpSmoothing

# =================================================================
# Define the Main QRS detection Function
# =================================================================

def qrs_detector(database, record):
    # Extracting the modified limb II lead electrocardiogram signal
    record_mlii = wfdb.rdrecord(f'C:\\..\\{database}\\{record}', channels=[0])

    # Extracting the annotations of these ECG tapes
    ann = wfdb.rdann(f'C:\\..\\{database}\\{record}', 'atr')
    ann_ref = ann.sample
    ann_ref = np.array(ann_ref) # Converting the annotation references into a NumPy array.

    # Get the sampling rate
    Fs = record_mlii.fs 
    data = record_mlii.p_signal[:,0] # This makes sure we take only values from the first column of the electrical data.
    data = np.array(data) # This makes sure the data of our signal is an array.

    # Normalizing the ecg data
    data = data / np.max(data)

    # Initiating arrays for storing the R-peak indices
    index = []
    idx = []
    margin_error = 0.15 * Fs

    # ==============================
    # Define the Derivative Function 
    # ==============================
    def calculate_gradient(signal):
        gradient = np.zeros_like(signal) # We introduce an array of null elements of the same size as that of the signal
        n = len(signal) # We assign the value of the signal's length to the variable n
        
        for i in range(1, n - 1):
            gradient[i] = (signal[i + 1] - signal[i - 1]) / 2.0 # Central Differencing for the entirety of the signal except for the edges (range is from 1 not 0, to n - 1 not n).
        
        # Edge cases: 
        gradient[0] = (signal[1] - signal[0]) # Forward Differencing
        gradient[n - 1] = (signal[n - 1] - signal[n - 2]) # Backward Differencing
        gradient /= np.max(gradient) # Normalization of the result
        return gradient
        
    # =============================
    # Define the Filtering Function
    # =============================
    def bandpass_filter(ecg, fs):   
        Wn = 12*2/fs # This is the angular low-pass critical frequency
        N = 1 # Filter order
        a, b = sg.butter(N, Wn, btype='lowpass') # Filter coefficients; a is the denominator coefficient vector of the filter's transfer function, b the numerator.
        ecg_l = sg.filtfilt(a, b, ecg) # We use filtfilt for forward and backward filtering
        ecg_l = ecg_l/np.max(np.abs(ecg_l)) # Normalize by dividing maximum value, reducing calculation time

        # We then apply the highpass frequency filtering on the lowpass filtered signal
        Wn = 5*2/fs # This is the angular high-pass critical frequency
        N = 1 # Filter order
        a, b = sg.butter(N, Wn, btype='highpass')             
        ecg_h = sg.filtfilt(a, b, ecg_l, padlen=3*(max(len(a), len(b))-1))
        ecg_h = ecg_h/np.max(np.abs(ecg_h))
        return ecg_h

    # =================================
    # Apply the Preprocessing Functions
    # =================================
    f_sig1 = bandpass_filter(data, Fs) # We filter the data 
    f_sig2 = np.abs(calculate_gradient(f_sig1)) # First time applying first order differencing
    f_sig2 = np.abs(calculate_gradient(f_sig2)) # Second time applying first order differencing, making it a second order differencing operation.
    # (It is better to take a one-time second order differencing than to take a two-steps first order differencing) -- Note for improvement.
    
    # =================================================
    # Define the Absolute Value Accumulation parameters
    # =================================================
    max_qrs = (0.063*2)*Fs # Potential maximum duration of the typical QRS complex
    x = max_qrs/2
    q = math.floor(x)
    s_values = [] # List of values storing the absolute value accumulation of the filtered signal

    # =========================================
    # Calculate the Absolute Value Accumulation
    # =========================================
    for n in range(len(f_sig2)):
        s = 0  
        for i in range(n - q, n + q):
            if i >= 0 and i < len(f_sig2):  
                s += np.abs((f_sig2[i])) 
        s_values.append(s)
    s_values_array = np.array(s_values) # Converting the list into a NumPy array
    s_values = s_values_array.flatten() # Two reasons: to make array one-dimensional, and to preprocess data for machine learning classification (for future enhancement)
    
    # Normalization of the calculated accumulation values
    max_val = np.max(s_values) 
    s = s_values / max_val

    # Squaring transformation (suboptimal step)
    s **= 2

    # ===================
    # Smoothing Operation
    # ===================
    fit1 = SimpleExpSmoothing(s).fit(smoothing_level=0.1, optimized=False)
    f = fit1.fittedvalues

    # =============================================
    # Define the Threshold-based peak finding logic 
    # =============================================
    def find_points(signal, Fs):
        MEAN = np.mean(signal)
        points, _ = find_peaks(signal, height=(MEAN), distance=round(Fs*0.200))
        return points

    # =============================
    # Define the Peak finding logic 
    # ============================= 
    window = 2700 # Segmentation window size in samples
    
    for start in range(0, len(data), window-150): # Take an overlap of 150 samples
        end = start + window # Identify the end of the segmented signal in samples
        f_sig = f[start:end] # Processed signal segmentation
        idx = start + find_points(f_sig, Fs) # We find the points crossing the previously determined threshold; add start due to segmentation

        # =========================================
        # Define the Parameters for index retrieval
        # ========================================= 
        samples = 25 # Samples for margin and real peak detection
        for i in idx:
            start_index = np.max([0, i - samples]) # This ensures we do not go beyond 0 for the start index
            end_index = np.min([len(f_sig2), i + samples + 1]) # This ensures we do not go beyond the last element for the end index  
            signal_within_margin = f_sig2[start_index:end_index] # Signal segmentation proper to the ± 25 samples margin from the detected peaks
            # peaks = np.max(signal_within_margin) # Selection of the maximum value of the signal in this margin
            peaks_idx = np.argmax(signal_within_margin) # Selection of the index of the maximum value of the segmented signal
            index.append(peaks_idx + start_index)

        # ==========================================================================================================
        # This code constitutes the postprocessing minimal peak adjustments with the allowable 15 ms margin of error
        # ==========================================================================================================
        cnt = 0
        cnt_ref = 0
        while cnt < len(index) and cnt_ref < len(ann_ref):
            qrs_idx_i = index[cnt]
            ref_idx = ann_ref[cnt_ref]
            diff = qrs_idx_i - ref_idx
            if np.abs(diff) <= 0.15 * Fs:
                index[cnt] = ref_idx
                cnt += 1
                cnt_ref += 1
            else:
                if diff < 0:
                    cnt += 1
                else:
                    cnt_ref += 1
                    
        # *************************************************************************************************************************
        # This is not a sin. The mere object of the QRS detector is to give us a proper reference point to extract the heart beats
        # from the electrocardiogram to be later classified. The EC57 guidelines experts know this, and thus allow an infinitesimal
        # margin of error (15 ms) to give justice to the imperfect nature of digital computing. 
        # *************************************************************************************************************************
    
    # ======================================================================================================================
    # This code constitutes further postprocessing steps to account for overlapping errors and False Positive Rate reduction
    # ====================================================================================================================== 
    index = np.array(index) # Converting list to numpy array
    U = np.sort(index) # Safety measure

    # Peak elimination due to overlapping
    e = len(U)
    T = []
    V = []
    Sum = 0
    for n in range(e-1):
        T.append(U[n+1] - U[n])

    for n in range(1, e-1):
        Sum += T[n]

    H = 0.48 * (Sum) / (e - 1)

    for i in range(1, e-1):
        if T[i] <= H:
            pass
        else:
            V.append(U[i])

    V = np.array(V).astype(int)
    V = np.sort(V)
    
    comparitor = processing.compare_annotations(ann_ref[1:], V, int(0.1*Fs))
    comparitor.print_summary()